# Hierarchical Clustering - UPGMA

---
## Learning Objectives

1. Implement a distance-based phylogeny neighbor joining algorithm
1. Understanding of hierarchical clustering
1. Manipulation of arrays and lists in Python


---
## Imports
You will need to install a two new packages to properly render your phylogenetic trees. Using your preferred package manager, install both [`biopython`](https://biopython.org/) and `matplotlib` into your virtual environments.

For example:
```bash
$ micromamba create -n module06 -c bioconda -c conda-forge python numpy matplotlib biopython
```

In [13]:
from typing import List, Tuple, Dict
from io import StringIO

from pyparsing import empty

from matrix_building import *
from pairwise_scorers import *
from scoring_manual_smith_waterman import *
import matplotlib.pyplot as plt
from Bio import Phylo
import numpy as np
import textdistance

---
## Background

Today we will implement a distance-based phylogenetic tree construction method using the Neighbor-Joining (NJ) algorithm and Smith-Waterman local alignment scores. Unlike UPGMA (Unweighted Pair Group Method using Arithmetic averages), NJ produces unrooted trees and does not assume a constant evolutionary rate across lineages, making it more biologically realistic for analyzing sequence relationships.

We will use HIV-1 reverse transcriptase sequences to construct our phylogenetic tree. The Smith-Waterman algorithm will be used to generate pairwise local alignment scores, which will then be converted into distances for tree construction. This approach is particularly suitable for HIV sequence analysis as it:
1. Handles sequence variations effectively through local alignment
2. Accounts for potential rate heterogeneity across different viral strains
3. Does not assume a molecular clock

The ultimate output will be an unrooted phylogenetic tree representing the evolutionary relationships between the HIV-1 sequences, visualized using the ete3 library. This method provides insights into viral diversity and evolutionary patterns while avoiding the assumptions of simpler hierarchical clustering approaches.

The key innovations of this implementation are:
- Use of Smith-Waterman for sensitive local alignment scoring
- Implementation of Neighbor-Joining for unrooted tree construction
- More biologically realistic evolutionary model

## Neighbor-Joining Algorithm

**Input**: Distance matrix $D$ for $n$ sequences

**Initialization**:
- Let $n$ be the number of sequences
- Assign each sequence $i$ to its own leaf node
- Initialize tree $T$ with leaf nodes

**Iteration**:
While $n > 2$:
1. Calculate Q-matrix where:
   $Q(i,j) = (n-2)d(i,j) - \sum_{k=1}^n d(i,k) - \sum_{k=1}^n d(j,k)$

2. Find pair $(i,j)$ with minimum $Q(i,j)$

3. Calculate branch lengths:
   $d_i = \frac{d(i,j) + (r_i - r_j)/(n-2)}{2}$
   $d_j = d(i,j) - d_i$
   where $r_i = \sum_{k=1}^n d(i,k)$

4. Create new node $k$
   - Add branches from $k$ to $i$ and $j$ with lengths $d_i$ and $d_j$

5. Update distances to remaining nodes $x$:
   $d(k,x) = \frac{d(i,x) + d(j,x) - d(i,j)}{2}$

6. Remove nodes $i$ and $j$
7. Add node $k$ to active nodes
8. $n = n - 1$

**Termination**:
When $n = 2$ with remaining nodes $i$ and $j$:
- Add final branch between $i$ and $j$ with length $d(i,j)$
- Return unrooted tree $T$

---
## Distance metrics for comparing sequences

### Smith-Waterman
This was previously implemented. Feel free to import if you know how...or Copy & Paste.



In [14]:
class TreeNode:
    def __init__(self, children=None, edge_weights=None, id=None):
        self.id = id
        self.edge_weights = edge_weights
        self.children = children
    def add_children(self, children, edge_weights):
        self.children.extend(children)
        self.edge_weights.extend(edge_weights)
    def walk(self):
        pass

In [15]:
fasta_file = fasta_to_Dict("data/lafayette_SARS_RT.fasta")
D,_ = build_distance_matrix(fasta_file)
active_layer = [TreeNode(key) for key in fasta_file] # list of the current active modes

#while len(active_layer) > 2:
if True:
    Q=D_to_Q(D)
    i,j = np.unravel_index(np.argmin(Q), Q.shape)
    r_i = D[i,:].sum()
    r_j = D[j,:].sum()

    d_ik = (D[i,j] + (r_i - r_j) / (D.shape[0]-2))/2
    d_jk = D[i, j] - d_ik

    d_ij = D[i,j]
    #d_kx = np.zeros(shape=D.shape[0])
    d_kx = (D[i,:] + D[j,:] - d_ij)/2


    D = np.vstack([D, d_kx])
    d_kx = np.concatenate([d_kx,[0]])
    D = np.hstack([D, d_kx.reshape(-1,1)])

    D = np.delete(D, [i,j], 0)
    D = np.delete(D, [i,j], 1)

    node_i = active_layer.pop(i)
    node_j = active_layer.pop(j)
    active_layer.append(TreeNode(children=[node_i, node_j], edge_weights=[d_ik, d_jk]))



    print(D)
    print(D.shape)





    #print(d_kx)








[[ 0.  1.  7. 23. 14.  6. 18. 15. 10. 14.  5.  9.  6. 17.  8.  8. 24.  9.
   8.]
 [ 1.  0.  6. 22. 13.  5. 17. 14.  9. 13.  4.  8.  5. 16.  7.  7. 23.  8.
   7.]
 [ 7.  6.  0. 22. 16.  9. 20. 18. 12. 15.  7. 11.  9. 11. 11. 11. 26. 12.
   2.]
 [23. 22. 22.  0. 20. 21. 20. 24. 21. 23. 19. 19. 21. 23. 23. 22. 23. 23.
  23.]
 [14. 13. 16. 20.  0. 13. 14. 19. 15. 17. 13. 14. 14. 21. 15. 16. 18. 14.
  17.]
 [ 6.  5.  9. 21. 13.  0. 14. 11.  6. 10.  3.  7.  2. 13.  5.  4. 22.  4.
  10.]
 [18. 17. 20. 20. 14. 14.  0. 17. 14. 17. 15. 13. 16. 19. 15. 17. 17. 16.
  21.]
 [15. 14. 18. 24. 19. 11. 17.  0. 14. 19. 12. 13. 13. 21. 12. 15. 25. 13.
  19.]
 [10.  9. 12. 21. 15.  6. 14. 14.  0. 13.  7.  8.  8. 14.  9. 10. 22. 10.
  13.]
 [14. 13. 15. 23. 17. 10. 17. 19. 13.  0. 11. 12. 12. 16. 13. 14. 24. 14.
  15.]
 [ 5.  4.  7. 19. 13.  3. 15. 12.  7. 11.  0.  6.  3. 13.  6.  5. 21.  7.
   8.]
 [ 9.  8. 11. 19. 14.  7. 13. 13.  8. 12.  6.  0.  7. 15.  9.  9. 21. 11.
  12.]
 [ 6.  5.  9. 21. 14.  2. 16

In [16]:
def neighbor_joining(
    distance_matrix: np.ndarray,
    labels: List[str]
) -> str:
    """Implements Neighbor-Joining algorithm for phylogenetic tree construction.

    Args:
        distance_matrix (np.ndarray): Distance matrix from Smith-Waterman scores
        labels (List[str]): Sequence identifiers

    Returns:
        str: Newick format tree string

    Examples:
        >>> seqs = read_fasta("lafayette_SARS_RT.fasta")
        >>> mat, ids = build_distance_matrix(seqs)
        >>> tree = neighbor_joining(mat, ids)
        >>> tree.startswith("((DM1:0.002,DM2:0.002)")  # Start of Newick string
        True
        >>> tree.count(",")  # Number of separators in tree
        18
    """
    pass

In [17]:
def plot_tree(newick_tree: str) -> None:
    """Plots a phylogenetic tree from a Newick string using Biopython and Matplotlib.

    The tree is rendered in a rectangular layout with smaller leaf labels and
    added margins to reduce overlap between labels and branches.

    Args:
        newick_tree (str): Tree in Newick format

    Returns:
        None: Displays the plotted tree
    """
    handle = StringIO(newick_tree)
    tree = Phylo.read(handle, "newick")

    fig, ax = plt.subplots(figsize=(8, 10))

    Phylo.draw(
        tree,
        axes=ax,
        do_show=False,
        label_func=lambda clade: clade.name if clade.is_terminal() else None,
    )
    for text in ax.texts:
        text.set_fontsize(6)              # shrink labels
    ax.margins(x=0.1, y=0.05)             # extra padding around tree

    plt.tight_layout()
    plt.show()


In [18]:
if __name__ == "__main__":
    # Read HIV RT sequences
    sequences = fasta_to_Dict("data/lafayette_SARS_RT.fasta")
    
    # Build distance matrix using Smith-Waterman
    dist_matrix, seq_ids = build_distance_matrix(sequences)
    
    # Generate unrooted tree using Neighbor-Joining
    tree = neighbor_joining(dist_matrix, seq_ids)
    
    # Plot tree
    plot_tree(tree)

ValueError: There are no trees in this file.